## Text input

https://platform.openai.com/docs/models

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import create_agent

agent = create_agent(
    model='groq:openai/gpt-oss-20b',
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [3]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

**The Capital of the Moon: Selene Prime**

---

### Overview

Selene Prime—often simply called “The City”—is the political, cultural, and economic heart of the lunar civilization. It sits in the heart of Mare Tranquillitatis, the “Sea of Tranquility,” a vast basalt plain that offers a relatively flat, stable foundation for large‑scale construction. The city’s name, derived from the Greek goddess of the Moon, reflects its status as the beacon of lunar society.

---

### Geography & Architecture

- **Domed Enclaves**  
  The city is a series of interconnected domes, each about 200 meters in diameter, constructed from regolith‑reinforced polymer composites. These domes maintain Earth‑like atmospheric pressure, temperature, and humidity, creating micro‑environments that support both human life and lunar flora.

- **Central Spire**  
  Rising 120 meters above the city’s surface, the Central Spire houses the Lunar Assembly Hall, the Executive Pavilion, and the Observatory Tower. Its façade i

## Image input

In [4]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [5]:
print(uploader.value)

()


In [7]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [11]:
print("Image input is not available with the current Groq models for this API key.")
print("Use a vision-capable provider for this section; the Groq text agent cannot accept image data.")

Image input is not available with the current Groq models for this API key.
Use a vision-capable provider for this section; the Groq text agent cannot accept image data.


## Audio input

In [12]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 50/50 [00:05<00:00,  9.80it/s]

Done.


In [13]:
from groq import Groq

transcription = Groq().audio.transcriptions.create(
    file=("recording.wav", wav_bytes),
    model="whisper-large-v3-turbo",
    response_format="text",
)

question = HumanMessage(
    content=f"Here is the transcript of an audio file:\n\n{transcription}\n\nSummarize it."
)
response = agent.invoke({"messages": [question]})
print(response['messages'][-1].content)

**Summary:**  
The audio contains a brief French plea: “S’il vous plaît. Merci. Mesdames et messieurs, nous aurions besoin d’une personne.” In English, this translates to: “Please. Thank you. Ladies and gentlemen, we would need a person.” The speaker is politely requesting assistance from the audience.
